# Prática: Introdução a Redes Neurais
# Insper AI
---

O objetivo desta atividade é compreender a arquitetura e o funcionamento interno de uma **Rede Neural (Multilayer Perceptron - MLP)**, implementando manualmente o processo de **Forward Propagation** usando apenas NumPy em um dataset realista (Fashion MNIST).

### Contexto
Anteriormente, exploramos a Regressão Logística - um modelo linear que consegue classificar apenas dados linearmente separáveis. Agora daremos o próximo passo: **Redes Neurais**, que podem aprender padrões complexos e não-lineares nos dados.

### Etapas
1. ReLU
2. Softmax
3. O processo de inferência (Forward Propagation)
4. Previsões e Avaliação
5. Comparação com TensorFlow/Keras
6. Conclusão

### Descrição do Dataset
O dataset **Fashion MNIST** é uma coleção de imagens de roupas em escala de cinza, com 60.000 imagens de treinamento e 10.000 imagens de teste. Cada imagem tem 28x28 pixels e pertence a uma das 10 categorias de roupas, como camisetas, calças, sapatos, etc.

É um dataset muito utilizado para testar modelos de ML, pois é uma alternativa mais desafiadora ao clássico MNIST de dígitos manuscritos.

O objetivo é classificar cada peça de roupa corretamente com base na imagem.

---

### Antes de Começar

Rode para sincronizar as dependências:

```bash
uv sync
```

---

## Carregando dados e bibliotecas

Nessa atividade, usaremos as seguintes bibliotecas:
- **pandas**: Biblioteca para manipulação e análise de dados estruturados.
- **numpy**: Biblioteca para computação numérica eficiente.
- **tensorflow/keras**: Usaremos apenas para carregar o dataset Fashion MNIST.
- **matplotlib**: Biblioteca para visualização de dados.

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Carregar os dados (como você já fez)
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

### Visualizando uma imagem do dataset
A imagem abaixo mostra alguns exemplos de imagens do dataset Fashion MNIST e suas respectivas classes.

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'{class_names[y_train[i]]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

---

## Implementação

Você vai escrever todas as funções necessárias para implementar o forward propagation e, só depois, aplicar isso no dataset.

**Ao final de cada célula, já existem testes simples para validar a implementação**.

### Tarefa 1: implemente a função ReLU em Python usando NumPy.
A função deve receber um valor real ou um array NumPy de qualquer shape e retornar um novo valor ou array em que cada elemento é o máximo entre 0 e o valor original.
$$\text{ReLU}(z) = \max(0, z)
$$

**Dica**: use a função `np.maximum`.

In [ ]:
def relu(z):
    """
    Args:
    z: valor real ou array NumPy de qualquer shape
    """
    # Seu código aqui
    

In [ ]:
# TESTE (Rode a célula)
test_array = np.array([-3, -1, 0, 1, 3])
print("Array Input:", test_array)
print("ReLU Output:", relu(test_array))
print("Excpected Output: [0 0 0 1 3]")

test_value = -5
print("\nValue Input:", test_value)
print("ReLU Output:", relu(test_value))
print("Excpected Output: 0")

### Tarefa 2: implemente a função Softmax em Python usando NumPy
Implemente a função softmax em Python usando NumPy. Para manter consistência com o restante do notebook, assuma que a função receberá um **batch 2D** de logits com shape `(n_amostras, n_classes)` e retornará um array de probabilidades com o mesmo shape.

Mesmo que exista apenas uma amostra, represente a entrada como um batch de tamanho 1, por exemplo com shape `(1, n_classes)`.

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}
$$

Onde:

- $z_i$: o logit da classe $i$, isto é, o score bruto produzido pela camada de saída para essa classe.
- $i$: o índice da classe cuja probabilidade estamos calculando.
- $j$: um índice auxiliar que percorre todas as classes no denominador, para normalizar os logits e fazer as probabilidades somarem 1.

**Dicas**:
- Use `np.exp` para calcular a exponencial.
- Use `np.sum` para calcular a soma dos exponenciais.
- Adicione uma pequena constante (ex: 1e-15) ao denominador para evitar divisão por zero.
- Use o parâmetro `axis = 1` no `np.sum` e `np.max`, pois cada linha representa uma amostra e cada coluna representa uma classe.
- Se quiser testar uma única amostra, faça `reshape(1, -1)` antes de chamar a função.

In [ ]:
def softmax(z):
    """
    Args:
    z: array de shape (n_amostras, n_classes) com os logits de cada amostra
    """

    # Estabiliza a entrada (NÃO MEXER)
    z_estavel = z - np.max(z, axis=1, keepdims=True)

    # Seu código aqui
    

In [ ]:
# TESTE (Rode a célula)

# Batch de teste com 2 amostras e 3 classes
test_batch = np.array([
    [1.0, 2.0, 3.0],
    [3.0, 2.0, 1.0]
])

print("Batch Input:", test_batch)
print("Shape do Input:", test_batch.shape)

print("\nSoftmax Output:\n", softmax(test_batch))
print("Expected Output:\n [[0.09003057 0.24472847 0.66524096]\n [0.66524096 0.24472847 0.09003057]]")

## O processo de inferência (Forward Propagation)

### O que é Forward Propagation?

**Forward Propagation** (Propagação Direta) é o processo pelo qual uma entrada percorre toda a rede neural, camada por camada, até produzir uma saída final. É literalmente como os dados "fluem para frente" através da rede.

### Como os Neurônios se Conectam

Matematicamente, cada camada realiza uma **transformação linear seguida de ativação**:

$$\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \cdot \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$
$$\mathbf{a}^{[l]} = f(\mathbf{z}^{[l]})$$

Onde:
- $\mathbf{W}^{[l]}$: Matriz de pesos da camada $l$
- $\mathbf{b}^{[l]}$: Vetor de bias da camada $l$
- $\mathbf{a}^{[l-1]}$: Ativações da camada anterior
- $f()$: Função de ativação (ReLU, Softmax, etc.)

Pense nisso como o seguinte:
1. Cada neurônio calcula uma soma ponderada das entradas (com pesos e bias)
2. Aplica a função de ativação para introduzir não-linearidade
3. Passa a ativação para a próxima camada -> Cada saída, de cada neurônio, torna-se uma entrada para todos os neurônios na próxima camada. Dessa forma, todos os neurônios estão interconectados (totalmente conectados).

#### O Processo Passo a Passo

Vamos rastrear como uma imagem de 28×28 pixels passa pela nossa rede:

**Passo 1: Preparação da Entrada**
```
Entrada original: Matriz 28×28 (imagem)
Entrada processada: Vetor 784×1 (flatten + normalização)
```

**Passo 2: Primeira Camada Oculta**
```
z₁ = W₁ × entrada + b₁     # Transformação linear
a₁ = ReLU(z₁)              # Função de ativação
```

**Passo 3: Segunda Camada Oculta**
```
z₂ = W₂ × a₁ + b₂          # Usa saída da camada anterior
a₂ = ReLU(z₂)              # Função de ativação
```

**Passo 4: Camada de Saída**
```
z₃ = W₃ × a₂ + b₃          # Transformação final
a₃ = Softmax(z₃)           # Probabilidades das classes
```

#### Dimensões das Matrizes

Para nossa arquitetura (784 → 128 → 64 → 10):

- **W₁**: 128 × 784 (cada linha = pesos de um neurônio)
- **b₁**: 128 × 1 (um bias por neurônio)
- **W₂**: 64 × 128
- **b₂**: 64 × 1
- **W₃**: 10 × 64
- **b₃**: 10 × 1

#### Representação Matemática Completa

$$\mathbf{a}^{[0]} = \text{entrada normalizada}$$
$$\mathbf{z}^{[1]} = \mathbf{W}^{[1]} \mathbf{a}^{[0]} + \mathbf{b}^{[1]}$$
$$\mathbf{a}^{[1]} = \text{ReLU}(\mathbf{z}^{[1]})$$
$$\mathbf{z}^{[2]} = \mathbf{W}^{[2]} \mathbf{a}^{[1]} + \mathbf{b}^{[2]}$$
$$\mathbf{a}^{[2]} = \text{ReLU}(\mathbf{z}^{[2]})$$
$$\mathbf{z}^{[3]} = \mathbf{W}^{[3]} \mathbf{a}^{[2]} + \mathbf{b}^{[3]}$$
$$\mathbf{a}^{[3]} = \text{Softmax}(\mathbf{z}^{[3]})$$

**Resultado**: $\mathbf{a}^{[3]}$ é um vetor de 10 probabilidades que somam 1!

### Implementação

#### Primeiro problema: como usar os dados de imagem?

A princípio, imagens, em escala de cinza, são representadas como matrizes 2D (altura x largura). No entanto, a maioria dos modelos de ML, incluindo MLPs, espera que os dados sejam fornecidos em formato vetorial (1D).

Para resolver isso, precisamos "achatar" cada imagem 2D em um vetor 1D. Como as imagens são de 28x28 pixels, cada uma deve ser convertida em um vetor de 784 (28x28) elementos.

#### Tarefa 3: transforme as imagens 2D em vetores 1D usando NumPy.
Gere dois novos arrays `X_train_flat` e `X_test_flat` com as imagens achatadas.

**Dicas**:
- Use a função `reshape` do NumPy.
- A função `reshape` pode receber `-1` como um dos argumentos para inferir automaticamente a dimensão correta.

In [ ]:
# Seu código aqui

### Agora sim, vamos implementar o Forward Propagation!

#### Tarefa 4: implemente o Forward Propagation em Python usando NumPy.
A função deve receber:
- `X`: array de shape (n_amostras, n_features) com as entradas (um batch de imagens achatadas)
- `weights`: lista de matrizes com os pesos de cada camada
- `biases`: lista de vetores com os biases de cada camada

E retornar:
- `output`: array de shape `(n_amostras, n_classes)` com as probabilidades de cada classe para cada amostra do batch

**Dicas**:
- Normalize as entradas dividindo por 255.0.
- Use um loop para iterar sobre as **camadas**. (SÓ SOBRE AS CAMADAS. NÃO FAÇA CÁLCULOS USANDO LOOPS)
- Use a função ReLU para as camadas ocultas e Softmax para a camada de saída.
- Use vetorização! Lembre-se de que a multiplicação de matrizes em NumPy é feita com o operador `@` ou a função `np.dot`.

**Dica crítica**:
- Com os pesos fornecidos neste notebook, a forma vetorizada correta é `np.dot(a, W.T) + b.T`, pois `a` está em shape `(n_amostras, n_features)` e cada `W` está em shape `(n_neuronios, n_features_entrada)`.
- `W.T` quer dizer **"a matriz W transposta"**


In [7]:
def forward_propagation(X, weights, biases):
    """
    Args:
    X: array de shape (n_amostras, n_features_entrada) com as entradas
    weights: lista de matrizes com os pesos de cada camada [W1, W2, ...]
    biases: lista de vetores com os biases de cada camada [b1, b2, ...]
    """
    # Normalize as entradas aqui


    # Use um Loop para iterar sobre as camadas ocultas (todas, menos a última) aqui
    # DICA: com os shapes deste notebook, a transformação linear é np.dot(a, W.T) + b.T


    # Faça o processamento da camada de saída (soma ponderada) aqui
    # DICA: o índice do último elemento de uma lista sempre é -1
    # DICA: aqui a transformação linear também segue a forma np.dot(a, W.T) + b.T


    # Ativação final com softmax aqui

    return
    

In [ ]:
# TESTE (Rode a célula)
X_teste = np.array([[255.0, 0.0], [0.0, 255.0]])
W_teste = [
    np.array([[1.0, 0.0], [0.0, 1.0]]),
    np.array([[1.0, 0.0], [0.0, 1.0]])
]
b_teste = [
    np.array([[0.0], [0.0]]),
    np.array([[0.0], [0.0]])
]

print("Forward Output:\n", forward_propagation(X_teste, W_teste, b_teste))
print("Expected Output:\n [[0.73105858 0.26894142]\n [0.26894142 0.73105858]]")

## Previsões e Avaliação

### Previsões
Após implementar o Forward Propagation, podemos usar a saída da rede para fazer previsões. A previsão para cada amostra é a classe com a maior probabilidade.

Rode a célula abaixo para carregar os pesos e biases pré-treinados:

In [ ]:
try:
    W1 = np.load('pesos/w1.npy')
    b1 = np.load('pesos/b1.npy')
    W2 = np.load('pesos/w2.npy')
    b2 = np.load('pesos/b2.npy')
    W3 = np.load('pesos/w3.npy')
    b3 = np.load('pesos/b3.npy')
    
    W = [W1, W2, W3]
    b = [b1, b2, b3]

    print("Pesos e biases carregados com sucesso!")
    print(f"Formato de W1: {W1.shape}")
    print(f"Formato de W2: {W2.shape}")
    print(f"Formato de W3: {W3.shape}")

except FileNotFoundError:
    print("Erro: A pasta 'pesos/' com os arquivos .npy não foi encontrada.")
    print("Certifique-se de ter baixado e colocado a pasta no mesmo diretório deste notebook.")

#### Tarefa 5: implemente a previsão e a avaliação

Use a função de Forward Propagation para obter as probabilidades e depois determine a classe prevista para cada imagem no conjunto de teste. Imprima a acurácia do modelo.

**Dicas**:
- Use `np.argmax` para encontrar o índice da classe com a maior probabilidade. Use o parâmetro `axis=1` para trabalhar com batches.
- Use a função de Forward Propagation que você implementou anteriormente.

In [ ]:
# Fazer previsões no conjunto de teste aqui


# Converter as probabilidades em classes previstas aqui


# Avaliar a acurácia aqui


Valor esperado de acurácia no conjunto de teste: **82.18%**

## Comparação com TensorFlow/Keras

A célula abaixo roda o mesmo modelo, implementado com um framework de alto nível (TensorFlow/Keras), para validar que nossa implementação manual está correta.

Apenas rode e compare com a seu resultado final.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Criar o modelo com a mesma arquitetura
model = Sequential([
    Flatten(input_shape=(28, 28)),  # Camada de entrada (flatten)
    Dense(128, activation='relu'),  # Primeira camada oculta
    Dense(64, activation='relu'),   # Segunda camada oculta
    Dense(10, activation='softmax') # Camada de saída
])

# Configurar os pesos e biases manualmente
model.layers[1].set_weights([W1.T, b1.flatten()])
model.layers[2].set_weights([W2.T, b2.flatten()])
model.layers[3].set_weights([W3.T, b3.flatten()])

# Compilar o modelo
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Avaliar o modelo no conjunto de teste
test_loss, test_accuracy = model.evaluate(X_test / 255.0 , y_test, verbose=0)
print(f"Acurácia no conjunto de teste: {test_accuracy * 100:.2f}%")

## Conclusão

Neste notebook, implementamos manualmente o processo de inferência de um MLP, o **Forward Propagation**, utilizando apenas NumPy. O resultado foi validado contra uma implementação equivalente em um framework de alto nível, confirmando a correção do nosso código.

**Recapitulando:**

* **Anatomia do MLP:** A estrutura de uma rede foi detalhada, compreendendo o papel da **Camada de Entrada** (representando as features), das **Camadas Ocultas** (aprendendo representações complexas) e da **Camada de Saída** (produzindo o resultado final para o número de classes).

* **O Papel das Funções de Ativação:** A necessidade de **Funções de Ativação não-lineares (ReLU)** foi estabelecida como o componente que permite à rede aprender padrões complexos. A **Softmax** foi apresentada como a solução padrão para converter os scores brutos da camada de saída em uma distribuição de probabilidade multiclasse.

* **Forward Propagation Desmistificado:** O processo de "previsão" de uma rede neural foi esclarecido. Entendemos que ele é uma sequência determinística de operações de álgebra linear (`Wx + b`) seguidas pela aplicação das funções de ativação, propagando a informação da entrada até a saída.

* **Conexão Teoria-Código:** A relação direta entre a notação matemática de uma rede neural e sua implementação prática com operações de matrizes em NumPy foi solidificada.